# Phase 3 — LoRA Rank Ablation (r=8/16/32)

Design (unchanged from v1 for comparability): 2,500-example subset, 2 epochs,
alpha=32, dropout=0.05 — only r varies. Parallel plan: r=8 and r=16 run
simultaneously (one per T4), then r=32.

**How to run:** execute Cells 1–5 interactively. If Cell 5's projection is
acceptable (< ~10 hrs), do **Save Version → Save & Run All (Commit)** and go
to sleep — the background run executes everything headless. Retrieve outputs
tomorrow from the notebook's *Output* tab.

In [ ]:
# CELL 1 — config (v1 fix #7: os.environ only). NOTE: no global CUDA_VISIBLE_DEVICES
# here — the orchestrator pins each training process to its own GPU.
import os
os.environ["REPO_URL"]        = "https://github.com/Rick-Roy-JC/text-to-sql-qlora-v2.git"
os.environ["DATA_DATASET"]    = "/kaggle/input/datasets/aritraroy3/spider-prepared-v2"
os.environ["CHECKPOINT_ROOT"] = "/kaggle/working/checkpoints"

In [ ]:
# CELL 2 — hardware tripwire
!nvidia-smi --query-gpu=index,name,compute_cap --format=csv
# Expect TWO rows of: Tesla T4, 7.5. If P100 appears -> STOP, fix accelerator.

In [ ]:
# CELL 3 — pinned installs (from notebooks/VERSIONS.txt, smoke-test-verified)
%pip install -q unsloth==2026.6.9 trl==0.24.0 peft==0.19.1 transformers==5.5.0
import unsloth, trl, peft, transformers, torch
print("unsloth", unsloth.__version__, "| trl", trl.__version__,
      "| peft", peft.__version__, "| transformers", transformers.__version__,
      "| torch", torch.__version__)

In [ ]:
# CELL 4 — repo + data (idempotent: safe on re-run and in background mode)
import os
%cd /kaggle/working
if not os.path.exists("text-to-sql-qlora-v2"):
    !git clone $REPO_URL
%cd text-to-sql-qlora-v2
!git pull
!mkdir -p data && cp $DATA_DATASET/*.jsonl data/
!ls -la data/

In [ ]:
# CELL 5 — timing probe: 15 real optimizer steps of r=8 at FULL ablation config.
# No checkpoint residue (save_steps=50 > 15). Prints sec/step; we project from it.
!python src/train.py --rank 8 --subset 2500 --epochs 2 --max_steps 15 \
    --output_root /tmp/probe

import math
STEPS_PER_RANK = math.ceil(2500 / 8) * 2          # ~626 optimizer steps
SEC_PER_STEP   = float(input("enter the sec/step printed above: "))
per_rank_h  = STEPS_PER_RANK * SEC_PER_STEP / 3600
wall_h      = 2 * per_rank_h            # parallel r8+r16, then r32
print(f"per rank ~{per_rank_h:.1f} h | projected wall time ~{wall_h:.1f} h "
      f"(+ ~15 min overhead)")
print("DECISION: <10 h -> commit this notebook as a background run."
      " >10 h -> tell Claude; we split ranks across sessions instead.")

In [ ]:
# CELL 6 — ORCHESTRATOR: r8 on GPU0 + r16 on GPU1 in parallel, then r32.
# In a committed background run this cell does the night shift.
import os, subprocess, time
ROOT = os.environ["CHECKPOINT_ROOT"]
os.makedirs(ROOT, exist_ok=True)

def launch(rank, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    log = open(f"{ROOT}/r{rank}.log", "w")
    p = subprocess.Popen(
        ["python", "src/train.py", "--rank", str(rank),
         "--subset", "2500", "--epochs", "2", "--output_root", ROOT],
        stdout=log, stderr=subprocess.STDOUT, env=env)
    print(f"launched r={rank} on GPU {gpu} (pid {p.pid}) -> {ROOT}/r{rank}.log")
    return p

t0 = time.time()
p8, p16 = launch(8, 0), launch(16, 1)
codes = {"r8": p8.wait(), "r16": p16.wait()}
print(f"[{(time.time()-t0)/3600:.1f} h] r8/r16 done, exit codes: {codes}")
p32 = launch(32, 0)
codes["r32"] = p32.wait()
print(f"[{(time.time()-t0)/3600:.1f} h] all done, exit codes: {codes}")
assert all(c == 0 for c in codes.values()), f"a run failed: {codes} — check logs"

In [ ]:
# CELL 7 — results table + package adapters for download
import json, glob, os, re
ROOT = os.environ["CHECKPOINT_ROOT"]
print(f"{'rank':<6}{'final eval_loss':<18}")
for r in (8, 16, 32):
    states = sorted(glob.glob(f"{ROOT}/r{r}/checkpoint-*/trainer_state.json"),
                    key=lambda p: int(re.search(r'checkpoint-(\d+)', p).group(1)))
    if not states:
        print(f"r={r:<4}NO CHECKPOINTS — see {ROOT}/r{r}.log"); continue
    hist = json.load(open(states[-1]))["log_history"]
    evals = [h["eval_loss"] for h in hist if "eval_loss" in h]
    print(f"r={r:<4}{evals[-1]:<18.4f}" if evals else f"r={r:<4}no eval logged")

!cd $CHECKPOINT_ROOT && zip -rq /kaggle/working/ablation_adapters.zip r8/adapter_final r16/adapter_final r32/adapter_final *.log
!ls -la /kaggle/working/ablation_adapters.zip
print("Download this zip from the Output tab; commit adapters + logs to git.")